# 이진 분류 실험 ⑨ — GNN 구조 장치 비교 (reverse MP · port numbering · ego ID)
피처는 **공통 17종(내 버전 20열)** 으로 고정하고, Multi-GNN 의 세 장치를 **하나씩 켜 가며** GINe 성능이 얼마나 오르는지 본다. HI-Large, 시드 1개.

| 조건 | reverse MP | port numbering | ego ID | 비고 |
|---|:-:|:-:|:-:|---|
| `plain` | ✕ | ✕ | ✕ | 정방향 메시지만 쓰는 기본 GINe |
| `rev` | ○ | ✕ | ✕ | + 역방향 메시지 전달 |
| `rev_port` | ○ | ○ | ✕ | + `in_port` · `out_port` |
| `rev_port_ego` | ○ | ○ | ○ | + ego ID (= 01 ~ 06 과 같은 구성) |

켜는 순서는 Multi-GNN 논문(Egressy et al., 2024) 의 ablation 표와 같다 (GIN → +ReverseMP → +Ports → +EgoIDs).

**끄는 방법** (`bc_lib.Config` 의 스위치 — 이번에 추가)
- `use_reverse_mp=False`: `rev_to` 이웃을 뽑지 않고(이웃 수 0), 모델도 `rev` 쪽 층을 만들지 않는다
- `use_ports=False`: `in_port` · `out_port` 열을 0 으로 둔다 (z-norm 뒤의 0 = 평균값이라 정보가 없다. 모델 구조는 그대로)
- `use_ego=False`: ego ID 열을 0 으로 둔다

**피처** — 결제방식(`payment_format`) 과 절대 시각(`timestamp`) 은 넣지 않는다 (07 · 08 결과)

| 묶음 | 열 | 출처 |
|---|---|---|
| 금액 2 | `amount_received`, `received_currency` | 01 |
| 같은 쌍 이력 4 | `pair_prior_count`, `pair_gap_minutes`, `pair_count_1h`, `reverse_history_count` | 02 |
| 1시간 건수 2 | `src_out_1h`, `dst_in_1h` | 03 |
| 계좌 이력 · 평소 대비 12 | `hist_out/in`, `hist_uniq_out/in`, `active_days`, `rate_vs_prior`, `minute_burst`, `gap_minutes` (송신 · 수신) | 06 |

- 계좌 누적 값(`hist_*`, `active_days`) 은 **거래마다 그 전날까지** 센 값을 엣지에 붙인 것이다. 공유 그래프에서는 노드 값이 계좌당 하나뿐이라 날짜별 값을 노드에 넣을 수 없어서 엣지에 싣는다. 08 의 v10 노드 피처(구간 끝 기준) 와 달리 미래 정보가 없다
- 학습 설정은 06 과 같다 (이웃 [100, 100], 배치 2048, 에폭 = 256 배치, 검증 4 에폭마다, patience 8, 검증 AP 로 선택). 마지막 조건 `rev_port_ego` 는 06 의 GINe 에서 피처만 바꾼 것이다
- **시드가 하나다.** 06 의 GINe 는 최고점(28 에폭, 검증 AP 0.588) 뒤에도 검증 AP 가 0.569 ~ 0.586 사이를 오갔다. 조건 간 차이가 0.02 안쪽이면 판단하지 않는다
- 예상 시간: 06(예전 서버) 은 에폭당 171초 · 60 에폭 학습 3.9시간 + 테스트 전체 추론 2.3시간이었다 → 조건 4개에 하루 안팎. `plain` 은 역방향 이웃을 안 뽑아 더 빠르다

## 0. 작업 준비

In [31]:
import sys, json, time, gc, dataclasses
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L

In [32]:
# ---- 실험 설정 ------------------------------------------------------------
SMOKE = False                     # True: 에폭 2 · 에폭당 8 배치 · 검증/테스트 일부만으로 끝까지 한 번 (에러 잡기용). 결과는 gnn_structure_smoke/ 에 따로
RUN_NAME = 'gnn_structure' + ('_smoke' if SMOKE else '')
cfg = B.Config(dataset='HI-Large', smoke=SMOKE, eval_batch_size=2048)   # 06 과 같은 설정. 구조 스위치는 조건마다 dataclasses.replace 로 바꿔 넣는다
B.seed_all(cfg.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = cfg.out_dir(RUN_NAME)

AMOUNT_FEATURES = ['amount_received', 'received_currency']
PAIR_FEATURES = ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'reverse_history_count']
ACT_FEATURES = ['src_out_1h', 'dst_in_1h']                # 03 의 8개 중 공통(1시간 건수) 2개만
HIST_FEATURES = ['src_hist_out', 'src_hist_uniq_out', 'src_active_days', 'src_rate_vs_prior', 'src_minute_burst', 'src_gap_minutes',
                 'dst_hist_in', 'dst_hist_uniq_in', 'dst_active_days', 'dst_rate_vs_prior', 'dst_minute_burst', 'dst_gap_minutes']
BASE_FEATURES = AMOUNT_FEATURES + PAIR_FEATURES + ACT_FEATURES + HIST_FEATURES      # 공통 17종의 내 버전 (20열)
GNN_FEATURES = BASE_FEATURES + B.PORT_FEATURES                                      # 피처 행렬 열 순서 (22). port 는 맨 뒤 두 열
PORT_COLS = [GNN_FEATURES.index('in_port'), GNN_FEATURES.index('out_port')]
PATHS = {'pair': cfg.feat_dir / 'pair_history.parquet', 'act': cfg.feat_dir / 'account_activity.parquet', 'hist': cfg.feat_dir / 'account_history.parquet'}
for p_ in PATHS.values():
    assert p_.exists(), f'앞 단계 parquet 이 없다: {p_}'

# 조건 — 논문 표 순서대로 하나씩 켠다
CONDS = {
    'plain':        dict(use_reverse_mp=False, use_ports=False, use_ego=False),
    'rev':          dict(use_reverse_mp=True,  use_ports=False, use_ego=False),
    'rev_port':     dict(use_reverse_mp=True,  use_ports=True,  use_ego=False),
    'rev_port_ego': dict(use_reverse_mp=True,  use_ports=True,  use_ego=True),
}
DEVICE, OUT_DIR, len(BASE_FEATURES), len(GNN_FEATURES)

(device(type='cuda'),
 PosixPath('/home/donghyeon/workspace/money_laundry_ai_/code/binary_classification/results/HI-Large/gnn_structure'),
 20,
 22)

## 1. 데이터 준비 — 거래 전체 + port + 앞 단계 피처
전부 `edge_id` 순 parquet 이라 조인 없이 열만 붙인다 (06 과 같은 방식)

In [ ]:
con = L.connect(threads=8, memory='12GB')
currency_codes, format_codes = B.code_tables(con, cfg)
PORTS_PATH = B.ensure_ports(con, cfg)               # features_ports/ports.parquet — 있으면 건너뜀

In [ ]:
con.close(); con = L.connect(threads=8, memory='3GB')
con.register('currency_codes', currency_codes); con.register('format_codes', format_codes)   # 코드표는 연결마다 다시 등록
tx = B.load_tx_lowmem(con, cfg, port_path=PORTS_PATH)                     # 기본 9열 + port 2열
tx = B.attach_parquet(tx, PATHS['pair'], PAIR_FEATURES, dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['act'], ACT_FEATURES, dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['hist'], HIST_FEATURES, dtype=np.float16)
B.trim('(피처 붙인 후)'); tx.shape

In [ ]:
# 라벨별 평균 — 피처가 제대로 붙었나 (06 의 표와 방향이 같아야 한다)
tx.groupby('y')[PAIR_FEATURES + ACT_FEATURES + HIST_FEATURES].mean().T.rename(columns={0: 'normal', 1: 'laundering'}).round(3)

## 2. z-norm (학습 구간 통계) + 표본

In [ ]:
in_train = B.window_mask(tx, cfg.train_window)
FEAT_PATH = cfg.feat_dir / 'feat16_gnn_structure.npy'                     # 22열 float16 memmap (약 7.9GB). smoke 도 같은 파일
feat_z, znorm_table = B.znorm_features(tx, GNN_FEATURES, in_train, out_path=FEAT_PATH)   # tx 에서 피처 열은 떨어짐
B.trim('(z-norm 후)'); znorm_table

In [ ]:
for name, w in [('train', cfg.train_window), ('val', cfg.val_window), ('test', cfg.test_window)]:
    m = B.window_mask(tx, w); assert m.sum() > 0, f'{name} 구간 {w} 에 거래가 없음'
    print(f'{name:5s} {w}  거래 {m.sum():>12,}  세탁 {tx.loc[m, "y"].sum():>8,}  비율 {tx.loc[m, "y"].mean()*100:.3f}%')

In [ ]:
val_df = B.val_sample(con, cfg.val_window, cfg.val_neg_rate)        # 검증: 양성 전부 + 정상 2% 가중 표본 (지금까지와 같은 파일)
test_df = B.test_full(con, cfg.test_window)                         # 테스트: 구간 전체
in_degree = B.recipient_indegree(con, cfg)                           # 분석용
con.close()
if cfg.smoke:                                                        # 에러 잡기용: 검증 1/20, 테스트 앞 20만 건만
    val_df = val_df.iloc[::20].reset_index(drop=True); test_df = test_df.iloc[:200_000].reset_index(drop=True)

all_edge_ids = tx['edge_id'].to_numpy()
val_idx = B.rows_of(all_edge_ids, val_df['edge_id'].to_numpy()); test_idx = B.rows_of(all_edge_ids, test_df['edge_id'].to_numpy())
assert (tx['y'].to_numpy()[val_idx] == val_df['y'].to_numpy()).all()
del all_edge_ids
train_rows = np.flatnonzero(in_train)
len(train_rows), val_df.shape, test_df.shape

## 3. 그래프 저장소 — 피처 행렬은 GPU 로
06 과 같다. 22열 float16 ≈ 7.9GB 를 GPU 에 올리고 배치 변환이 거기서 뽑는다

In [ ]:
store = B.GraphStore(tx, feat_z, cfg, train_rows=train_rows, val_idx=val_idx, test_idx=test_idx, port_cols=PORT_COLS)
del tx, feat_z; B.trim('(tx, feat_z 삭제 후)')
store.features_to_gpu(DEVICE)
print(f'GPU 할당 {torch.cuda.memory_allocated() / 1e9:.1f} GB'); B.trim('(피처 GPU 이동 후)')
store.edge_dim

## 4. 스위치 확인 — 조건마다 배치 하나
학습 전에 세 스위치가 실제로 먹는지 본다. 기대값:

| 조건 | rev_to 간선 | port 열 절댓값 합 | ego 합 |
|---|---|---|---|
| `plain` | 0 | 0 | 0 |
| `rev` | > 0 | 0 | 0 |
| `rev_port` | > 0 | > 0 | 0 |
| `rev_port_ego` | > 0 | > 0 | > 0 (target 양끝 계좌 수) |

`plain` 은 `rev` 쪽 층이 없어서 파라미터 수가 다른 조건보다 적다

In [ ]:
def set_cond(flags):
    # 조건의 스위치를 넣은 cfg 를 만들어 store 에 건다. 스위치는 loader 가 들고 있으므로, 올라와 있는 그래프는 두고 loader 만 다시 만든다
    cfg_c = dataclasses.replace(cfg, **flags)
    store.cfg = cfg_c
    cur = store.current
    if cur['graph'] is not None:
        cur['loader'] = store.make_loader(cur['graph'], cur['targets'], shuffle=(cur['name'] == 'train'))
    return cfg_c


rows = []
for name, flags in CONDS.items():
    cfg_c = set_cond(flags)
    batch = next(iter(store.use('train')))
    model = B.make_gine(store.edge_dim, cfg_c, edge_updates=False).to(DEVICE)
    with torch.no_grad():
        logits, _ = B.forward_batch(model, batch, DEVICE)
    rows.append({'조건': name, '노드': batch['node'].num_nodes, 'to 간선': batch[B.TO].num_edges, 'rev_to 간선': batch[B.REV].num_edges,
                 'port 절댓값 합': round(float(batch[B.TO].edge_attr[:, PORT_COLS].abs().sum()) + float(batch.target_attr[:, PORT_COLS].abs().sum()), 1),
                 'ego 합': int(batch['node'].x[:, 1].sum()), '파라미터': sum(p.numel() for p in model.parameters()), '로짓': tuple(logits.shape)})
    del model, batch, logits; torch.cuda.empty_cache()
check = pd.DataFrame(rows).set_index('조건'); check

In [ ]:
# 기대값과 다르면 여기서 멈춘다
assert check.loc['plain', 'rev_to 간선'] == 0 and check.loc['plain', 'port 절댓값 합'] == 0 and check.loc['plain', 'ego 합'] == 0
assert check.loc['rev', 'rev_to 간선'] > 0 and check.loc['rev', 'port 절댓값 합'] == 0 and check.loc['rev', 'ego 합'] == 0
assert check.loc['rev_port', 'port 절댓값 합'] > 0 and check.loc['rev_port', 'ego 합'] == 0
assert check.loc['rev_port_ego', 'ego 합'] > 0
print('스위치 확인 통과')

## 5. 학습 · 테스트 — 조건마다
조건 하나 = 학습(검증 AP 최고 체크포인트 저장) → 테스트 구간 전체 추론 → 결과 저장.
**끝난 단계는 건너뛴다**: `<조건>_history.csv` 가 있으면 학습을, `<조건>_test_scores.parquet` 가 있으면 테스트를 건너뛴다. 커널이 죽으면 0 ~ 4 절을 다시 돌리고 이 절을 실행하면 이어진다
(학습 도중에 죽은 조건은 처음부터 다시 학습한다)

In [ ]:
results_path = OUT_DIR / 'results.json'
results = json.loads(results_path.read_text(encoding='utf-8'))['results'] if results_path.exists() else {}


def run_cond(name, flags):
    cfg_c = set_cond(flags)
    B.seed_all(cfg.seed)                                    # 조건마다 같은 시드로 시작
    if (OUT_DIR / f'{name}_history.csv').exists():
        print(f'[{name}] 학습 끝난 것 있음 — 건너뜀')
    else:
        B.log(f'[{name}] 학습 시작  {flags}')
        model, _ = B.train_gnn(name, store, cfg_c, val_df, OUT_DIR, DEVICE, edge_updates=False, results=results)
        del model; gc.collect(); torch.cuda.empty_cache()
    if (OUT_DIR / f'{name}_test_scores.parquet').exists():
        print(f'[{name}] 테스트 점수 있음 — 건너뜀')
    else:
        B.test_gnn(name, store, cfg_c, test_df, OUT_DIR, DEVICE, False, results)
    return B.save_results(OUT_DIR, cfg, results, features=dict(gnn=GNN_FEATURES, conditions=CONDS))   # 조건 하나 끝날 때마다 저장


for name, flags in CONDS.items():
    compare = run_cond(name, flags)
compare

# 결과 — 저장된 점수 파일만으로 돌아간다
커널을 새로 켰으면 **0 절(설정 셀 2개)** 만 돌리고 여기부터. 재료는 `results/HI-Large/gnn_structure/`

## 6. 전체 지표 — 한 단계씩 켤 때의 변화

In [ ]:
ORDER = [c for c in CONDS if (OUT_DIR / f'{c}_test_scores.parquet').exists()]
cmp = pd.read_csv(OUT_DIR / 'compare.csv', index_col=[0, 1])
res = json.loads((OUT_DIR / 'results.json').read_text(encoding='utf-8'))['results']
tbl = pd.DataFrame({c: {'val_AP': cmp.loc[(c, 'val'), 'AP'], 'test_AP': cmp.loc[(c, 'test'), 'AP'], 'F1': cmp.loc[(c, 'test'), 'F1'],
                        'F1@0.5': cmp.loc[(c, 'test'), 'F1_at_0.5'], 'recall@1%': cmp.loc[(c, 'test'), 'recall_at_1pct'],
                        'unmatched@1%': cmp.loc[(c, 'test'), 'unmatched_recall_at_1pct']} for c in ORDER}).T
tbl.to_csv(OUT_DIR / 'summary.csv', encoding='utf-8-sig')
display(tbl.round(4))
print('앞 조건 대비 차이 (그 장치를 켜서 얻은 몫)'); display(tbl.diff().iloc[1:].round(4))

In [ ]:
# 학습 기록 — 최고 에폭, 돈 에폭 수, 학습 시간, 에폭당 시간, 파라미터 수 (reverse MP 를 켜면 이웃이 두 배라 느려진다)
pd.DataFrame({c: {k: res[c].get(k) for k in ('best_epoch', 'epochs_run', 'train_sec', 'epoch_sec', 'n_params', 'gpu_peak_gb')} for c in ORDER if c in res}).T

In [ ]:
# 학습 곡선 — 조건별 검증 AP. 출렁임 폭이 조건 간 차이보다 크면 그 차이는 믿지 않는다
fig, ax = plt.subplots(figsize=(8, 3.5))
for c in ORDER:
    h = pd.read_csv(OUT_DIR / f'{c}_history.csv').dropna(subset=['AP']); ax.plot(h['epoch'], h['AP'], marker='.', ms=4, lw=1, label=c)
ax.set_xlabel('epoch'); ax.set_ylabel('val AP'); ax.legend(); plt.tight_layout(); plt.savefig(OUT_DIR / 'curves.png', dpi=120); plt.show()

## 7. 패턴 기준 지표 — 08 과 같은 계산
무패턴 양성은 정답도 오답도 아닌 것으로 두고 알림에서 뺀다. 알림 수는 조건마다 같다 (테스트 전체 상위 0.1% · 1%)

In [ ]:
scores = {c: B.load_scores(OUT_DIR / f'{c}_test_scores.parquet') for c in ORDER}
ref_ids = scores[ORDER[0]]['edge_id'].to_numpy()
for c in ORDER:
    assert np.array_equal(scores[c]['edge_id'].to_numpy(), ref_ids), f'{c} 점수의 행 순서가 다르다'
lab = B.pattern_array(scores[ORDER[0]])                       # 'NORMAL' · 'UNMATCHED' · 패턴 이름
is_normal, is_un = lab == 'NORMAL', lab == 'UNMATCHED'
is_pat = ~is_normal & ~is_un
# 결제방식(ACH 여부) 은 평가에서만 쓴다
con = L.connect(threads=8, memory='3GB')
a, b = L.parse_window(cfg.test_window)
fmt = con.execute(f'''SELECT edge_id, (payment_format = 'ACH') AS is_ach FROM read_parquet({L.q(cfg.tx_path)})
                      WHERE day BETWEEN DATE {L.q(a)} AND DATE {L.q(b)} ORDER BY edge_id''').fetchnumpy()
con.close()
pos = np.minimum(np.searchsorted(fmt['edge_id'], ref_ids), len(fmt['edge_id']) - 1)
assert (fmt['edge_id'][pos] == ref_ids).all()
is_ach = np.asarray(fmt['is_ach'], dtype=bool)[pos]; del fmt, pos
print(f'테스트 {len(lab):,}건 · 패턴 {int(is_pat.sum()):,} · 무패턴 {int(is_un.sum()):,} · 정상 {int(is_normal.sum()):,}')

In [ ]:
rows = []
for c in ORDER:
    s = scores[c]['score'].to_numpy()
    r = {'조건': c}
    for budget in (0.001, 0.01):
        k = f'{budget * 100:g}%'
        hit = s >= np.quantile(s, 1 - budget)
        tp_pat, tp_un, fp = int((hit & is_pat).sum()), int((hit & is_un).sum()), int((hit & is_normal).sum())
        r.update({f'알림@{k}': int(hit.sum()), f'패턴R@{k}': tp_pat / max(int(is_pat.sum()), 1), f'패턴P@{k}': tp_pat / max(tp_pat + fp, 1),
                  f'무패턴R@{k}': tp_un / max(int(is_un.sum()), 1), f'정상오탐@{k}': fp, f'비ACH알림@{k}': int((hit & ~is_ach).sum())})
    rows.append(r)
pat_tbl = pd.DataFrame(rows).set_index('조건')
pat_tbl.to_csv(OUT_DIR / 'pattern_metrics.csv', encoding='utf-8-sig')
pat_tbl.round(4)

## 8. 패턴 유형별 · 수신 계좌 차수별

In [ ]:
# 패턴 유형별 recall — 상위 0.1% · 1% 컷과 확률 0.5. 어느 장치가 어느 패턴을 살리나
pat = pd.concat([B.pattern_recall(scores[c]).assign(cond=c) for c in ORDER])
pat.to_csv(OUT_DIR / 'analysis_pattern.csv', encoding='utf-8-sig')
order_p = [p for p in B.PATTERN_ORDER + ['ALL'] if p in pat.index]
for col in ('r@0.1%', 'r@1%', 'argmax'):
    print(f'--- {col}'); display(pat.reset_index().pivot(index='pattern_type', columns='cond', values=col).loc[order_p, ORDER].round(4))

In [ ]:
# 수신 계좌 in-degree 구간별 AP — 허브 구간에서 어느 장치가 도움이 되나
if 'test_df' not in globals() or 'in_degree' not in globals():
    con = L.connect(threads=8, memory='12GB')
    test_df = B.test_full(con, cfg.test_window); in_degree = B.recipient_indegree(con, cfg); con.close()
test_dst = pd.Series(test_df['dst'].to_numpy(), index=test_df['edge_id'].to_numpy()).reindex(ref_ids).to_numpy()   # 점수 행 순서에 맞춘 dst
deg = pd.concat([B.degree_stratified(scores[c], test_dst, in_degree).assign(cond=c) for c in ORDER])
deg.to_csv(OUT_DIR / 'analysis_degree.csv', encoding='utf-8-sig')
bins = [f'[{lo}-{hi}]' if hi < 10**9 else f'[{lo}+]' for lo, hi in B.DEGREE_BINS]
deg.reset_index().pivot(index='bin', columns='cond', values='AP').loc[[b for b in bins if b in deg.index], ORDER].round(4)

## 9. 읽는 법
- **6 절의 차이 표**: 각 행이 그 장치를 켜서 얻은 몫이다. 순서를 바꾸면 몫이 달라질 수 있다 (하나씩 쌓는 비교의 한계)
- **시드가 하나다.** 학습 곡선(6 절) 에서 검증 AP 가 출렁이는 폭을 먼저 본다. 조건 간 차이가 그 폭보다 작으면 "차이 없음" 으로 읽는다
- **7 절**: 같은 알림 수에서 패턴 precision 과 무패턴 recall 이 장치에 따라 어떻게 바뀌나. 08 의 트리(`base`: 패턴 R 0.962 · P 0.759 @0.1%) 는 v10 버전 피처(구간 끝 기준 노드 값 포함) 라서 절대값을 그대로 비교하면 안 된다
- **8 절** — 확인할 가설: port 는 같은 계좌의 여러 상대를 구분해 주므로 FAN-IN/OUT · SCATTER/GATHER 에서, ego ID 는 target 양끝을 표시해 주므로 CYCLE 에서 오를 것
- `plain` 은 역방향 층이 없어 파라미터 수가 적다. `rev` 와의 차이에는 "역방향 메시지" 와 "층이 늘어난 것" 이 같이 들어 있다
- port 를 끈 조건도 `in_port` · `out_port` 열 자리는 남아 있다 (값만 0). 입력 차원은 네 조건이 같다
- 차이가 작게 나오면: 이 피처 셋(쌍 이력 · 계좌 이력) 이 구조 정보를 이미 담고 있어서일 수 있다. 06 에서 같은 피처로 트리(LightGBM 테스트 AP 0.634) 가 GINe(0.583) 를 넘었다